In [24]:
import numpy as np
import os
from typing import List, Tuple
from ipynb.fs.full.audio_parser import audio_convert, spectrogram_conversion
from ipynb.fs.full.fingerprint_maker import generate_fingerprints

In [25]:
#need to finish commenting

def create_database(): #highkey redundant if not assigning independent string
    fingerprint_database = dict()
    return fingerprint_database


def add_fingerprints(database: dict, song_id: str, fingerprints: list):
    #adds songs from our library to a dictionary of fingerprints
    for (fm, fn, dt), tm in fingerprints:
        if (fm, fn, dt) not in database:
                database[(fm, fn, dt)] = []
        database[(fm, fn, dt)].append((song_id, tm))

def query_database(database: dict, query_fingerprints: list):

    #returns an array of how many matches with each entry in the dict

    match_counts = {}

    for (fm, fn, dt), tq in query_fingerprints:
        fingerprint_key = (fm, fn, dt)

        if fingerprint_key in database:

            for song_id, tm in database[fingerprint_key]:
                time_offset = tm - tq

                if (song_id, time_offset) not in match_counts:
                    match_counts[song_id, time_offset] = 0

                match_counts[song_id, time_offset] += 1

    return match_counts


def get_best_match(match_counts):
    #returns song id and time offset of best match
    #need to add probability feature (if agreed on) using returned highest_count 
    #and remove time offset artifact from highest key. 
    if len(match_counts) == 0:
        return None
    
    highest_key = max(match_counts, key=match_counts.get)
    highest_count = match_counts[highest_key]
    return highest_key, highest_count


In [33]:
song_path = os.path.join("Music", "Billie Jean.wav") 
print(f"Processing: {song_path}...")

samples, sample_rate = audio_convert(song_path)

print("Samples shape:", samples.shape)
print("Sample rate:", sample_rate)
print("Max volume sample value:", np.max(np.abs(samples)))

log_spectro, extracted_peaks = spectrogram_conversion(samples, sample_rate)
print("Spectrogram shape:", log_spectro.shape)
print("Min/Max values in Spectrogram:", np.min(log_spectro), np.max(log_spectro))
print(f"Jesse's function extracted {len(extracted_peaks)} peaks.")

my_fingerprints = generate_fingerprints(extracted_peaks, fanout=3)
print(f"Successfully generated {len(my_fingerprints)} unique fingerprints!")
database = create_database()

#generating fingerprints for test sample
test_song_path = os.path.join("Music", "Billie Jean.wav")
test_samples, test_sample_rate = audio_convert(test_song_path)
test_log_spectro, test_extracted_peaks = spectrogram_conversion(test_samples, test_sample_rate)
test_fingerprints = generate_fingerprints(test_extracted_peaks, fanout=3)


add_fingerprints(database, song_id="Billie Jean", fingerprints = my_fingerprints)
match_counts = query_database(database, test_fingerprints)
best_match = get_best_match(match_counts)
print("Best match:", best_match)

print("Database fingerprints:", len(my_fingerprints))
print("Test fingerprints:", len(test_fingerprints))
print("Matches:", len(match_counts))

Processing: Music/Billie Jean.wav...
Samples shape: (4707909,)
Sample rate: 16000
Max volume sample value: 0.007598877
Spectrogram shape: (1025, 4596)
Min/Max values in Spectrogram: -100.0 -72.34395
Jesse's function extracted 273 peaks.
Successfully generated 813 unique fingerprints!
Best match: (('Billie Jean', 0), 813)
Database fingerprints: 813
Test fingerprints: 813
Matches: 233
